## Setup & Config

In [1]:
# RAG Canonical Examples Manager

import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

# Where your semantic jsonl files live
DATA_DIR = Path("data/semantic_chunks")   # change this if you like

CORE_PATH = DATA_DIR / "semantic_chunks.core.jsonl"
USAGE_PATH = DATA_DIR / "semantic_chunks.usage.jsonl"
QA_PATH = DATA_DIR / "semantic_chunks.qa.jsonl"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Core file:", CORE_PATH)
print("Usage file:", USAGE_PATH)
print("QA file:", QA_PATH)


Core file: data\semantic_chunks\semantic_chunks.core.jsonl
Usage file: data\semantic_chunks\semantic_chunks.usage.jsonl
QA file: data\semantic_chunks\semantic_chunks.qa.jsonl


## Low-level JSONL helpers

In [2]:
def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    """Append a single JSON object as one line to a .jsonl file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Read a .jsonl file into a list of dicts (empty list if missing)."""
    if not path.exists():
        return []
    records: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


## Examples SDK” (core / usage / QA writers)

In [3]:
def add_core_example(
    *,
    id: str,
    question: str,
    intent_type: str,
    expected_metrics: List[str],
    expected_dimensions: List[str],
    structured_query_spec: Dict[str, Any],
    reasoning_notes: str,
    edge_case_type: Optional[str] = None,
) -> Dict[str, Any]:
    """Append a canonical core example to semantic_chunks.core.jsonl."""
    record: Dict[str, Any] = {
        "id": id,
        "question": question,
        "intent_type": intent_type,
        "expected_metrics": expected_metrics,
        "expected_dimensions": expected_dimensions,
        "structured_query_spec": structured_query_spec,
        "reasoning_notes": reasoning_notes,
        "edge_case_type": edge_case_type,
        "created_at": datetime.utcnow().isoformat() + "Z",
    }
    append_jsonl(CORE_PATH, record)
    print(f"✅ Added core example {id}")
    return record


def add_usage_example(
    *,
    id: str,
    canonical_id: str,
    question: str,
    intent_type: Optional[str] = None,
    notes: Optional[str] = None,
) -> Dict[str, Any]:
    """Append a usage-paraphrase example to semantic_chunks.usage.jsonl."""
    record: Dict[str, Any] = {
        "id": id,
        "canonical_id": canonical_id,
        "question": question,
        "intent_type": intent_type,
        "notes": notes,
        "created_at": datetime.utcnow().isoformat() + "Z",
    }
    append_jsonl(USAGE_PATH, record)
    print(f"✅ Added usage example {id} → {canonical_id}")
    return record


def add_qa_example(
    *,
    id: str,
    canonical_id: str,
    question: str,
    expected_intent_type: str,
    expected_fact_tables: List[str],
    expected_derived_metrics: Optional[List[str]] = None,
    expected_joins: Optional[List[str]] = None,
    critical_assertions: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """Append a QA/eval example to semantic_chunks.qa.jsonl."""
    record: Dict[str, Any] = {
        "id": id,
        "canonical_id": canonical_id,
        "question": question,
        "expected_intent_type": expected_intent_type,
        "expected_fact_tables": expected_fact_tables,
        "expected_derived_metrics": expected_derived_metrics or [],
        "expected_joins": expected_joins or [],
        "critical_assertions": critical_assertions or [],
        "created_at": datetime.utcnow().isoformat() + "Z",
    }
    append_jsonl(QA_PATH, record)
    print(f"✅ Added QA example {id} → {canonical_id}")
    return record


## Simple diagnostics / validation helpers

In [4]:
def list_ids(path: Path, label: str, limit: int = 20):
    records = read_jsonl(path)
    print(f"{label}: {len(records)} records")
    for r in records[:limit]:
        print("-", r.get("id"))


def check_duplicate_ids(path: Path, label: str):
    records = read_jsonl(path)
    seen = {}
    dups = []
    for r in records:
        _id = r.get("id")
        if _id in seen:
            dups.append(_id)
        else:
            seen[_id] = True
    if not dups:
        print(f"✅ No duplicate IDs in {label}")
    else:
        print(f"⚠️ Duplicate IDs in {label}:", dups)


def summary():
    list_ids(CORE_PATH, "CORE", limit=10)
    list_ids(USAGE_PATH, "USAGE", limit=10)
    list_ids(QA_PATH, "QA", limit=10)
    print()
    check_duplicate_ids(CORE_PATH, "CORE")
    check_duplicate_ids(USAGE_PATH, "USAGE")
    check_duplicate_ids(QA_PATH, "QA")


In [28]:
# === Category 12: Edge & No-Data / Anti-Hallucination Cases ===
# Assumes:
# - add_core_example and read_jsonl are already defined
# - CORE_PATH is defined

existing_ids = {r["id"] for r in read_jsonl(CORE_PATH)}

def safe_add_core_example(*, id: str, **kwargs):
    if id in existing_ids:
        print(f"⏭️ Skipping {id} (already exists in core)")
        return
    record = add_core_example(id=id, **kwargs)
    existing_ids.add(id)
    return record


# 1) C12-GoogleCampaign-PerfNoAuctionInsight-001
# Campaigns with performance but no auction insight data.
structured_query_spec_C12_001 = {
    "primary_fact": "GoogleAdsCampaignPerformanceMetric",
    "secondary_facts": [
        {
            "fact": "GoogleAdsCampaignAuctionInsightMetric",
            "join_key": ["CampaignId"],
        }
    ],
    "grain": "campaign",
    "metrics": [
        {
            "name": "Cost",
            "table": "GoogleAdsCampaignPerformanceMetric",
            "aggregation": "sum",
            "source_column": "Cost",
        },
        {
            "name": "Impressions",
            "table": "GoogleAdsCampaignPerformanceMetric",
            "aggregation": "sum",
            "source_column": "Impressions",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "AccountId",  "table": "GoogleAdsCampaignPerformanceMetric"},
        {"name": "CampaignId", "table": "GoogleAdsCampaignPerformanceMetric"},
    ],
    "joins": [
        {
            "type": "left",
            "from": {"table": "GoogleAdsCampaignPerformanceMetric", "column": "CampaignId"},
            "to":   {"table": "GoogleAdsCampaignAuctionInsightMetric", "column": "CampaignId"},
        }
    ],
    "time_range": {
        "field": "CalendarId",
        "table": "GoogleAdsCampaignPerformanceMetric",
        "type": "relative",
        "value": "last_30_days",
    },
    "group_by": [
        "GoogleAdsCampaignPerformanceMetric.AccountId",
        "GoogleAdsCampaignPerformanceMetric.CampaignId",
    ],
    "filters": [],
    "post_aggregation_filters": [
        {
            "field": "GoogleAdsCampaignAuctionInsightMetric.CampaignId",
            "operator": "IS_NULL",
            "value": None,
        }
    ],
}

safe_add_core_example(
    id="C12-GoogleCampaign-PerfNoAuctionInsight-001",
    question=(
        "Which Google Ads campaigns had spend in the last 30 days but no auction insight data?"
    ),
    intent_type="diagnostic.data_gaps",
    expected_metrics=[
        "Cost",
        "Impressions",
    ],
    expected_dimensions=[
        "GoogleAdsCampaignPerformanceMetric.AccountId",
        "GoogleAdsCampaignPerformanceMetric.CampaignId",
    ],
    structured_query_spec=structured_query_spec_C12_001,
    reasoning_notes=(
        "Anti-hallucination pattern: detects campaigns where performance metrics exist but "
        "GoogleAdsCampaignAuctionInsightMetric has no rows. Uses a LEFT join and IS NULL filter "
        "on the auction table to surface gaps rather than fabricating auction metrics."
    ),
    edge_case_type="no_auction_insight_for_campaign",
)


# 2) C12-Events-WithOrdersButNoEventMap-002
# Events with orders but missing mapping in EventMap (no cluster/venue/category).
structured_query_spec_C12_002 = {
    "primary_fact": "EventOrderMetric",
    "grain": "event",
    "metrics": [
        {
            "name": "OrderRevenue",
            "table": "EventOrderMetric",
            "aggregation": "sum",
            "source_column": "OrderTotal",
        },
        {
            "name": "TicketsSold",
            "table": "EventOrderMetric",
            "aggregation": "sum",
            "source_column": "Quantity",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "EventId",   "table": "Event"},
        {"name": "EventName", "table": "Event"},
    ],
    "joins": [
        {
            "type": "inner",
            "from": {"table": "EventOrderMetric", "column": "EventId"},
            "to":   {"table": "Event",            "column": "EventId"},
        },
        {
            "type": "left",
            "from": {"table": "Event", "column": "EventId"},
            "to":   {"table": "EventMap", "column": "EventId"},
        },
    ],
    "time_range": {
        "field": "CalendarId",
        "table": "EventOrderMetric",
        "type": "relative",
        "value": "last_30_days",
    },
    "group_by": [
        "Event.EventId",
        "Event.EventName",
    ],
    "filters": [],
    "post_aggregation_filters": [
        {
            "field": "EventMap.EventId",
            "operator": "IS_NULL",
            "value": None,
        }
    ],
}

safe_add_core_example(
    id="C12-Events-WithOrdersButNoEventMap-002",
    question=(
        "In the last 30 days, which events have order revenue but no EventMap mapping "
        "(no cluster/venue/category assigned)?"
    ),
    intent_type="diagnostic.data_gaps",
    expected_metrics=[
        "OrderRevenue",
        "TicketsSold",
    ],
    expected_dimensions=[
        "Event.EventId",
        "Event.EventName",
    ],
    structured_query_spec=structured_query_spec_C12_002,
    reasoning_notes=(
        "Detects events that have EventOrderMetric activity but no corresponding EventMap row, "
        "meaning they cannot be rolled up to clusters/venues/categories. Uses LEFT join from Event "
        "to EventMap and IS NULL filter to flag missing mappings."
    ),
    edge_case_type="missing_event_mapping",
)


# 3) C12-Funnel-ImpressionsWithoutAdsSource-003
# Funnel activity with no matching ads traffic source (likely direct / organic).
structured_query_spec_C12_003 = {
    "primary_fact": "FunnelMetric",
    "grain": "day_page_type",
    "metrics": [
        {
            "name": "ImpressionCount",
            "table": "FunnelMetric",
            "aggregation": "sum",
            "source_column": "ImpressionCount",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "CalendarId", "table": "FunnelMetric"},
        {"name": "PageType",   "table": "FunnelMetric"},
    ],
    "joins": [
        {
            "type": "left",
            "from": {"table": "FunnelMetric", "column": "TrafficSourceId"},
            "to":   {"table": "GoogleAdsFunnelMetricTrafficSource", "column": "TrafficSourceId"},
        },
        {
            "type": "left",
            "from": {"table": "FunnelMetric", "column": "TrafficSourceId"},
            "to":   {"table": "MicrosoftAdsFunnelMetricTrafficSource", "column": "TrafficSourceId"},
        },
    ],
    "time_range": {
        "field": "CalendarId",
        "table": "FunnelMetric",
        "type": "relative",
        "value": "last_7_days",
    },
    "group_by": [
        "FunnelMetric.CalendarId",
        "FunnelMetric.PageType",
    ],
    "filters": [],
    "post_aggregation_filters": [
        {
            "field": "GoogleAdsFunnelMetricTrafficSource.TrafficSourceId",
            "operator": "IS_NULL",
            "value": None,
        },
        {
            "field": "MicrosoftAdsFunnelMetricTrafficSource.TrafficSourceId",
            "operator": "IS_NULL",
            "value": None,
        },
    ],
}

safe_add_core_example(
    id="C12-Funnel-ImpressionsWithoutAdsSource-003",
    question=(
        "Over the last 7 days, by day and page type, where do we have funnel impressions that "
        "are not attributed to any Google or Microsoft Ads traffic source?"
    ),
    intent_type="diagnostic.attribution_gaps",
    expected_metrics=[
        "ImpressionCount",
    ],
    expected_dimensions=[
        "FunnelMetric.CalendarId",
        "FunnelMetric.PageType",
    ],
    structured_query_spec=structured_query_spec_C12_003,
    reasoning_notes=(
        "Shows funnel activity that cannot be tied back to a paid traffic source in "
        "GoogleAdsFunnelMetricTrafficSource or MicrosoftAdsFunnelMetricTrafficSource. "
        "Useful to avoid hallucinating paid attribution where none exists, and to highlight "
        "direct/organic buckets."
    ),
    edge_case_type="funnel_without_paid_source",
)


# 4) C12-Events-FutureNoInventoryNoExchange-004
# Future events with zero inventory and zero exchange metrics (completely unlisted).
structured_query_spec_C12_004 = {
    "primary_fact": "EventSummary",
    "secondary_facts": [
        {"fact": "EventInventoryMetric", "join_key": ["EventId"]},
        {"fact": "StandardExchangeMetric", "join_key": ["EventId"]},
    ],
    "grain": "event",
    "metrics": [
        {
            "name": "ListingCount",
            "table": "EventInventoryMetric",
            "aggregation": "sum",
            "source_column": "ListingCount",
        },
        {
            "name": "ExchangeRevenue",
            "table": "StandardExchangeMetric",
            "aggregation": "sum",
            "source_column": "ExchangeRevenue",
        },
        {
            "name": "ExchangeOrders",
            "table": "StandardExchangeMetric",
            "aggregation": "sum",
            "source_column": "ExchangeOrders",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "EventId",            "table": "Event"},
        {"name": "EventName",          "table": "Event"},
        {"name": "EventStatus",        "table": "EventSummary"},
        {"name": "EventTimestampUtc",  "table": "EventSummary"},
        {"name": "IsFutureEvent",      "table": "EventSummary"},
    ],
    "joins": [
        {
            "type": "inner",
            "from": {"table": "EventSummary", "column": "EventId"},
            "to":   {"table": "Event",        "column": "EventId"},
        },
        {
            "type": "left",
            "from": {"table": "EventSummary", "column": "EventId"},
            "to":   {"table": "EventInventoryMetric", "column": "EventId"},
        },
        {
            "type": "left",
            "from": {"table": "EventSummary", "column": "EventId"},
            "to":   {"table": "StandardExchangeMetric", "column": "EventId"},
        },
    ],
    "time_range": {
        "field": "EventTimestampUtc",
        "table": "EventSummary",
        "type": "relative",
        "value": "next_60_days",
    },
    "group_by": [
        "Event.EventId",
        "Event.EventName",
        "EventSummary.EventStatus",
        "EventSummary.EventTimestampUtc",
        "EventSummary.IsFutureEvent",
    ],
    "filters": [
        {
            "field": "IsFutureEvent",
            "table": "EventSummary",
            "operator": "=",
            "value": True,
        }
    ],
    "post_aggregation_filters": [
        {
            "field": "ListingCount",
            "operator": "=",
            "value": 0,
        },
        {
            "field": "ExchangeRevenue",
            "operator": "=",
            "value": 0,
        },
        {
            "field": "ExchangeOrders",
            "operator": "=",
            "value": 0,
        },
    ],
}

safe_add_core_example(
    id="C12-Events-FutureNoInventoryNoExchange-004",
    question=(
        "Which future events in the next 60 days have no listings and no exchange revenue or orders?"
    ),
    intent_type="diagnostic.data_gaps",
    expected_metrics=[
        "ListingCount",
        "ExchangeRevenue",
        "ExchangeOrders",
    ],
    expected_dimensions=[
        "Event.EventId",
        "Event.EventName",
        "EventSummary.EventStatus",
        "EventSummary.EventTimestampUtc",
        "EventSummary.IsFutureEvent",
    ],
    structured_query_spec=structured_query_spec_C12_004,
    reasoning_notes=(
        "Composite no-data scenario: future events that are completely unlisted both onsite and "
        "on exchanges. Teaches the model to check multiple fact tables before claiming an event "
        "has inventory or sales."
    ),
    edge_case_type="unlisted_future_events",
)


# 5) C12-GoogleCampaign-SpendNoOrdersNoExchange-005
# Campaigns with ad spend but no onsite orders and no exchange revenue.
structured_query_spec_C12_005 = {
    "primary_fact": "GoogleAdsCampaignPerformanceMetric",
    "secondary_facts": [
        {"fact": "EventOrderMetric", "join_key": ["CalendarId"]},
        {"fact": "StandardExchangeMetric", "join_key": ["CalendarId"]},
    ],
    "grain": "campaign",
    "metrics": [
        {
            "name": "AdCost",
            "table": "GoogleAdsCampaignPerformanceMetric",
            "aggregation": "sum",
            "source_column": "Cost",
        },
        {
            "name": "OnsiteOrderRevenue",
            "table": "EventOrderMetric",
            "aggregation": "sum",
            "source_column": "OrderTotal",
        },
        {
            "name": "ExchangeRevenue",
            "table": "StandardExchangeMetric",
            "aggregation": "sum",
            "source_column": "ExchangeRevenue",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "AccountId",  "table": "GoogleAdsCampaignPerformanceMetric"},
        {"name": "CampaignId", "table": "GoogleAdsCampaignPerformanceMetric"},
    ],
    "joins": [
        {
            "type": "left",
            "from": {"table": "GoogleAdsCampaignPerformanceMetric", "column": "CalendarId"},
            "to":   {"table": "EventOrderMetric", "column": "CalendarId"},
        },
        {
            "type": "left",
            "from": {"table": "GoogleAdsCampaignPerformanceMetric", "column": "CalendarId"},
            "to":   {"table": "StandardExchangeMetric", "column": "CalendarId"},
        },
    ],
    "time_range": {
        "field": "CalendarId",
        "table": "GoogleAdsCampaignPerformanceMetric",
        "type": "relative",
        "value": "last_30_days",
    },
    "group_by": [
        "GoogleAdsCampaignPerformanceMetric.AccountId",
        "GoogleAdsCampaignPerformanceMetric.CampaignId",
    ],
    "filters": [],
    "post_aggregation_filters": [
        {
            "field": "AdCost",
            "operator": ">",
            "value": 0,
        },
        {
            "field": "OnsiteOrderRevenue",
            "operator": "=",
            "value": 0,
        },
        {
            "field": "ExchangeRevenue",
            "operator": "=",
            "value": 0,
        },
    ],
}

safe_add_core_example(
    id="C12-GoogleCampaign-SpendNoOrdersNoExchange-005",
    question=(
        "In the last 30 days, which Google Ads campaigns had spend but generated zero onsite "
        "order revenue and zero exchange revenue?"
    ),
    intent_type="diagnostic.data_gaps",
    expected_metrics=[
        "AdCost",
        "OnsiteOrderRevenue",
        "ExchangeRevenue",
    ],
    expected_dimensions=[
        "GoogleAdsCampaignPerformanceMetric.AccountId",
        "GoogleAdsCampaignPerformanceMetric.CampaignId",
    ],
    structured_query_spec=structured_query_spec_C12_005,
    reasoning_notes=(
        "Campaign-level version of 'spend but no revenue' detection. Aligns GoogleAdsCampaignPerformanceMetric "
        "with EventOrderMetric and StandardExchangeMetric at the CalendarId level only, explicitly checking for "
        "zero revenue across both facts. Prevents the model from assuming revenue exists just because spend does."
    ),
    edge_case_type="ad_spend_without_any_revenue",
)


# 6) C12-Orders-DetailMissingCampaignAttribution-006
# Orders with missing Google/Microsoft campaign attribution.
structured_query_spec_C12_006 = {
    "primary_fact": "Order",
    "grain": "order",
    "metrics": [
        {
            "name": "OrderTotal",
            "table": "Order",
            "aggregation": "none",
            "source_column": "OrderTotal",
        },
    ],
    "derived_metrics": [],
    "dimensions": [
        {"name": "OrderId",          "table": "Order"},
        {"name": "OrderTimestampUtc","table": "Order"},
        {"name": "Quantity",         "table": "Order"},
        {"name": "BillingState",     "table": "Order"},
        {"name": "EventId",          "table": "Order"},
        {"name": "GoogleAdsCampaignId",   "table": "Order"},
        {"name": "MicrosoftAdsCampaignId","table": "Order"},
    ],
    "joins": [],
    "time_range": {
        "field": "OrderTimestampUtc",
        "table": "Order",
        "type": "relative",
        "value": "last_7_days",
    },
    "group_by": [],
    "filters": [
        {
            "field": "GoogleAdsCampaignId",
            "table": "Order",
            "operator": "IS_NULL",
            "value": None,
        },
        {
            "field": "MicrosoftAdsCampaignId",
            "table": "Order",
            "operator": "IS_NULL",
            "value": None,
        },
    ],
}

safe_add_core_example(
    id="C12-Orders-DetailMissingCampaignAttribution-006",
    question=(
        "In the last 7 days, list orders that have no Google or Microsoft campaign attribution, "
        "including order totals, quantity, billing state, and event."
    ),
    intent_type="diagnostic.attribution_gaps",
    expected_metrics=[
        "OrderTotal",
    ],
    expected_dimensions=[
        "Order.OrderId",
        "Order.OrderTimestampUtc",
        "Order.Quantity",
        "Order.BillingState",
        "Order.EventId",
        "Order.GoogleAdsCampaignId",
        "Order.MicrosoftAdsCampaignId",
    ],
    structured_query_spec=structured_query_spec_C12_006,
    reasoning_notes=(
        "Order-level attribution gap detector: finds orders where both GoogleAdsCampaignId and "
        "MicrosoftAdsCampaignId are NULL. Important to stop the model from attributing these orders "
        "to ads when no attribution is present in the data."
    ),
    edge_case_type="orders_without_campaign_attribution",
)

print("✅ Category 12 core examples added/checked.")


✅ Added core example C12-GoogleCampaign-PerfNoAuctionInsight-001
✅ Added core example C12-Events-WithOrdersButNoEventMap-002
✅ Added core example C12-Funnel-ImpressionsWithoutAdsSource-003
✅ Added core example C12-Events-FutureNoInventoryNoExchange-004
✅ Added core example C12-GoogleCampaign-SpendNoOrdersNoExchange-005
✅ Added core example C12-Orders-DetailMissingCampaignAttribution-006
✅ Category 12 core examples added/checked.


In [29]:
summary()

CORE: 72 records
- C1-GoogleCampaignDailyPerf-001
- C1-GoogleAdGroupDailyPerf-Account-002
- C1-MicrosoftCampaignDailyPerf-003
- C1-MicrosoftAdGroupAggregatePerf-004
- C1-GoogleCampaignDailyPerf-ByDateSynonym-005
- C1-GoogleCampaignAggregatePerf-AllAccounts-006
- C2-GoogleCampaignDaily-CR-CPA-RPC-001
- C2-GoogleCampaignAggregate-ROI-RPC-002
- C2-MicrosoftAdGroup-CPA-RPC-EdgeNoConversions-003
- C2-GoogleCampaign-CPA-ROI-RPC-SynonymROAS-004
USAGE: 0 records
QA: 0 records

✅ No duplicate IDs in CORE
✅ No duplicate IDs in USAGE
✅ No duplicate IDs in QA


## (Optional) Promote successful production runs

In [ ]:
LOG_PATH = Path("data/rag_usage_log.jsonl")

def iter_usage_log():
    if not LOG_PATH.exists():
        print("No usage log yet:", LOG_PATH)
        return
    with LOG_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def promote_successful_runs_to_core(prefix: str = "AUTO"):
    for row in iter_usage_log():
        if not row.get("success"):
            continue

        spec = row["structured_query_spec"]
        q = row["nl_question"]
        run_id = row.get("id") or datetime.utcnow().isoformat()

        core_id = f"{prefix}-{run_id}"

        add_core_example(
            id=core_id,
            question=q,
            intent_type=spec.get("intent_type", "unknown"),
            expected_metrics=[m.get("name") for m in spec.get("metrics", [])],
            expected_dimensions=[d for d in spec.get("group_by", [])],
            structured_query_spec=spec,
            reasoning_notes="Promoted from successful production run.",
            edge_case_type=None,
        )

# Example usage:
# promote_successful_runs_to_core()


## Spec Validator

In [31]:
# === Spec Validator for structured_query_spec vs semantic_schema ===
# Assumes:
# - read_jsonl is already defined
# - CORE_PATH points at semantic_chunks.core.jsonl
#
# You MUST set SCHEMA_PATH to your semantic_schema.json file location.

SCHEMA_PATH = "outputs/semantic_schema.json"  # <-- change if needed
SPEC_PATH = CORE_PATH                 # you can override to QA/usage file later

import json
from collections import defaultdict

# ---- load schema ----
with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)

tables_by_name = {t["logical_name"]: t for t in schema.get("tables", [])}
columns_by_table = {
    t["logical_name"]: {c["name"] for c in t.get("columns", [])}
    for t in schema.get("tables", [])
}

# Conceptual / virtual fields that are allowed to be "missing" in schema
VIRTUAL_FIELDS = {
    "HasInventoryFlag",  # conceptual flag from Category 7 example
}

def table_exists(table_name: str) -> bool:
    return table_name in tables_by_name

def column_exists(table_name: str, column_name: str) -> bool:
    if column_name in VIRTUAL_FIELDS:
        return True  # treat as allowed conceptual field
    cols = columns_by_table.get(table_name)
    if not cols:
        return False
    return column_name in cols

def _fmt_issue(level: str, message: str) -> str:
    return f"{level}: {message}"

def validate_structured_spec(spec: dict, example_id: str):
    issues = []

    # ----- primary fact -----
    primary = spec.get("primary_fact")
    if not primary:
        issues.append(_fmt_issue("ERROR", "missing primary_fact"))
    elif not table_exists(primary):
        issues.append(_fmt_issue("ERROR", f"primary_fact table '{primary}' not found in schema"))

    # ----- secondary_facts -----
    for sf in spec.get("secondary_facts", []):
        fact = sf.get("fact")
        if not fact:
            issues.append(_fmt_issue("ERROR", "secondary_facts entry missing 'fact'"))
            continue
        if not table_exists(fact):
            issues.append(_fmt_issue("ERROR", f"secondary_fact table '{fact}' not found in schema"))

    # ----- dimensions -----
    for dim in spec.get("dimensions", []):
        t = dim.get("table")
        name = dim.get("name")
        source_col = dim.get("source_column") or name
        if not t:
            issues.append(_fmt_issue("ERROR", f"dimension '{name}' missing table"))
            continue
        if not table_exists(t):
            issues.append(_fmt_issue("ERROR", f"dimension table '{t}' not found (dim='{name}')"))
            continue
        if source_col and not column_exists(t, source_col):
            issues.append(_fmt_issue(
                "WARNING",
                f"dimension column '{source_col}' not found on table '{t}' (dim='{name}')"
            ))

    # ----- metrics -----
    for m in spec.get("metrics", []):
        name = m.get("name")
        t = m.get("table") or primary
        source_col = m.get("source_column") or name
        if not t:
            issues.append(_fmt_issue("ERROR", f"metric '{name}' missing table (and no primary_fact?)"))
            continue
        if not table_exists(t):
            issues.append(_fmt_issue("ERROR", f"metric table '{t}' not found (metric='{name}')"))
            continue
        # Allow metrics that are clearly derived / do not directly map to columns
        if source_col and not column_exists(t, source_col):
            issues.append(_fmt_issue(
                "WARNING",
                f"metric column '{source_col}' not found on table '{t}' (metric='{name}')"
            ))

    # ----- joins -----
    for j in spec.get("joins", []):
        for side in ("from", "to"):
            side_obj = j.get(side) or {}
            t = side_obj.get("table")
            c = side_obj.get("column")
            if not t or not c:
                issues.append(_fmt_issue(
                    "ERROR",
                    f"join side '{side}' missing table or column: {side_obj}"
                ))
                continue
            if not table_exists(t):
                issues.append(_fmt_issue("ERROR", f"join {side} table '{t}' not found"))
                continue
            if not column_exists(t, c):
                issues.append(_fmt_issue(
                    "ERROR",
                    f"join {side} column '{c}' not found on table '{t}'"
                ))

    # ----- time_range field -----
    tr = spec.get("time_range")
    if tr:
        tr_table = tr.get("table") or primary
        tr_field = tr.get("field")
        if tr_field:
            if not table_exists(tr_table):
                issues.append(_fmt_issue(
                    "ERROR",
                    f"time_range table '{tr_table}' not found (field='{tr_field}')"
                ))
            elif not column_exists(tr_table, tr_field):
                issues.append(_fmt_issue(
                    "WARNING",
                    f"time_range field '{tr_field}' not found on table '{tr_table}'"
                ))

    return issues


# ---- run validator over all core examples ----
records = read_jsonl(SPEC_PATH)
spec_issues_by_id = {}
error_count = 0
warning_count = 0

for rec in records:
    ex_id = rec.get("id", "<no-id>")
    spec = rec.get("structured_query_spec")
    if not spec:
        # Not all records need to be specs; skip non-spec core chunks
        continue
    issues = validate_structured_spec(spec, ex_id)
    if issues:
        spec_issues_by_id[ex_id] = issues
        for msg in issues:
            if msg.startswith("ERROR"):
                error_count += 1
            elif msg.startswith("WARNING"):
                warning_count += 1

print("===== Spec Validation Summary =====")
print(f"Total records in file: {len(records)}")
print(f"Records with spec issues: {len(spec_issues_by_id)}")
print(f"Total ERROR messages:   {error_count}")
print(f"Total WARNING messages: {warning_count}")
print()

for ex_id, issues in spec_issues_by_id.items():
    print(f"--- {ex_id} ---")
    for msg in issues:
        print("  ", msg)
    print()


===== Spec Validation Summary =====
Total records in file: 72
Records with spec issues: 16
Total ERROR messages:   28
Total WARNING messages: 16

--- C1-GoogleCampaignDailyPerf-001 ---
   ERROR: dimension table 'DimCalendar' not found (dim='CalendarDate')
   ERROR: join to table 'DimCalendar' not found
   ERROR: time_range table 'DimCalendar' not found (field='CalendarDate')

--- C1-GoogleAdGroupDailyPerf-Account-002 ---
   ERROR: dimension table 'DimCalendar' not found (dim='CalendarDate')
   ERROR: join to table 'DimCalendar' not found
   ERROR: time_range table 'DimCalendar' not found (field='CalendarDate')

--- C1-MicrosoftCampaignDailyPerf-003 ---
   ERROR: dimension table 'DimCalendar' not found (dim='CalendarDate')
   ERROR: join to table 'DimCalendar' not found
   ERROR: time_range table 'DimCalendar' not found (field='CalendarDate')

--- C1-GoogleCampaignDailyPerf-ByDateSynonym-005 ---
   ERROR: dimension table 'DimCalendar' not found (dim='CalendarDate')
   ERROR: join to tab